# Building Footprint Extraction — Eğitim Notebook'u

Bu notebook şunları yapar:
1. `alexatanassov/building-footprint-extraction` reposunu klonlar
2. Bağımlılıkları kurar
3. **Massachusetts Buildings Dataset**'i resmi kaynaktan (cs.toronto.edu/~vmnih) indirir
4. Reponun kendi script'lerinin (`tile_generator.py`, `scripts/train_unet.py` vb.) gerçek `--help` çıktısını göstererek doğru parametrelerle eğitimi başlatır

> Not: Adım 4'teki parametreler tahmin değildir — script'lerin kendi `--help` çıktısından okunur. Çıktıyı görüp uygun komutu birlikte tamamlayacağız.

**Runtime > Change runtime type > GPU** seçili olduğundan emin olun.

## 1) Repoyu klonla ve yapısına bak

In [ ]:
!rm -rf building_footprint
!git clone https://github.com/faiggafarov/building_footprint.git
%cd building_footprint


In [ ]:
# README'nin tamamını gör (dataset klasör yapısı, kurulum, kullanım talimatları burada)
!cat README.md

## 2) Bağımlılıkları kur

In [ ]:
!pip install -q -r requirements.txt

## 3) Veri setini indir — Massachusetts Buildings Dataset

Kaynak: resmi Toronto Üniversitesi sayfası — https://www.cs.toronto.edu/~vmnih/data/mass_buildings/

Her split (`train`, `valid`, `test`) için ayrı `sat/` (uydu görüntüsü, .tiff) ve `map/` (bina maskesi, .tif) klasörleri var; indirme sayfaları birer dosya listesi (index.html) olduğu için `wget -r` ile o listedeki tüm dosyaları çekiyoruz.

In [ ]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

BASE = "https://www.cs.toronto.edu/~vmnih/data/mass_buildings"
DATA_ROOT = "raw_mass_buildings"

splits = {
    "train": ["sat", "map"],
    "valid": ["sat", "map"],
    # "test":  ["sat", "map"] # Sürətli test üçün testi yükləmirik
}

def download_split(split, kind, limit=None):
    index_url = f"{BASE}/{split}/{kind}/index.html"
    out_dir = f"{DATA_ROOT}/{split}/{kind}"
    os.makedirs(out_dir, exist_ok=True)
    
    try:
        r = requests.get(index_url, timeout=30)
        soup = BeautifulSoup(r.text, "html.parser")
        links = [a["href"] for a in soup.find_all("a", href=True) if a["href"].lower().endswith((".tif", ".tiff"))]
        
        # Yalnız ilk bir neçə faylı yükləyirik (Sürətli təlim üçün)
        if limit:
            links = links[:limit]
        print(f"{split}/{kind}: {len(links)} dosya tapıldı, yüklənir...")
        
        for href in links:
            file_url = urljoin(index_url, href)
            fname = os.path.join(out_dir, os.path.basename(href))
            if os.path.exists(fname):
                continue
            data = requests.get(file_url, timeout=60).content
            with open(fname, "wb") as f:
                f.write(data)
        return len(links)
    except Exception as e:
        print(f"Xəta baş verdi: {e}")
        return 0

for split, kinds in splits.items():
    for kind in kinds:
        download_split(split, kind, limit=None)

print("Yükləmə tamamlandı!")


## 4) Reponun beklediği veri klasör yapısını öğren

Repo README'sinde `datasets/` klasörünün "Custom PyTorch datasets and loaders" içerdiği yazıyor. Doğru klasör isimlendirmesini ve bekleneni tahmin etmek yerine, loader kodunun kendisine bakalım.

In [ ]:
!echo '--- datasets/ içeriği ---'
!ls -la datasets/
!echo
!echo '--- datasets/ altındaki python dosyalarının tamamı ---'
!find datasets -name '*.py' -exec echo '## {}' \; -exec cat {} \;

## 4) Massachusetts Verisini Modele Uygun Hale Getir (Tiling)

Repodaki `tile_generator.py` Vegas (GeoJSON) verisi için yazılmış. Massachusetts verisi (TIF) için özel bir tile oluşturucu script yazıp çalıştırıyoruz. Bu script görüntüleri 256x256 boyutunda kırpıp `train_unet.py`'nin beklediği `.npz` (C,H,W) formatında kaydeder.

In [ ]:
%%writefile mass_tile_generator.py
import os, glob
import numpy as np
from PIL import Image
from tqdm import tqdm

def process_massachusetts():
    sat_dir = "raw_mass_buildings/train/sat"
    map_dir = "raw_mass_buildings/train/map"
    out_img = "data/mass_tiles_npz/images"
    out_lbl = "data/mass_tiles_npz/labels"
    import shutil
    if os.path.exists(out_img): shutil.rmtree(out_img)
    if os.path.exists(out_lbl): shutil.rmtree(out_lbl)
    os.makedirs(out_img, exist_ok=True)
    os.makedirs(out_lbl, exist_ok=True)
    
    sat_files = sorted(glob.glob(f"{sat_dir}/*.tiff"))  # Bütün şəkilləri emal et
    for sat_path in tqdm(sat_files, desc="Tiling Images"):
        basename = os.path.basename(sat_path).replace(".tiff", ".tif")
        map_path = os.path.join(map_dir, basename)
        if not os.path.exists(map_path): continue
        
        img = np.array(Image.open(sat_path)) # H, W, 3
        mask = np.array(Image.open(map_path).convert('L')) # H, W (Grayscale)
        mask = (mask > 0).astype(np.uint8)
        
        H, W = img.shape[:2]
        size = 256
        for y in range(0, H - size + 1, size):
            for x in range(0, W - size + 1, size):
                img_tile = img[y:y+size, x:x+size]
                mask_tile = mask[y:y+size, x:x+size]
                if mask_tile.sum() == 0 and np.random.rand() > 0.1: continue # Boş tile'ların %90'ını at
                
                img_tile = img_tile.transpose(2, 0, 1) # C, H, W
                tile_name = f"{basename.replace('.tif','')}_{y}_{x}.npz"
                np.savez_compressed(os.path.join(out_img, tile_name), arr_0=img_tile)
                np.savez_compressed(os.path.join(out_lbl, tile_name), arr_0=mask_tile)

process_massachusetts()
print('Tiling işlemi bitti!')


In [ ]:
!python mass_tile_generator.py

## 5) Eğitim Scripti Hazırdır

Biz artıq `models/train_unet.py` faylını öz repository-mizdə Massachusetts dataset-ə və Early Stopping-ə uyğun düzəltmişik. Əlavə heç nəyə ehtiyac yoxdur!

## 6) Eğitimi Başlat

Artık modelimizi Massachusetts verisi ile eğitebiliriz.

In [ ]:
!python models/train_unet.py

## 7) Sonuçları Analiz Et (Loss ve Accuracy/Dice)

Eğitim sırasında kaydedilen log dosyasını oxuyaraq Train Loss, Validation Dice ve Validation IoU metriklerini analiz edirik.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

log_path = 'logs/unet_metrics.csv'
if os.path.exists(log_path):
    df = pd.read_csv(log_path)
    print("--- Eğitim Tablosu ---")
    display(df)
    
    plt.figure(figsize=(12, 5))
    
    # Loss Plot
    plt.subplot(1, 2, 1)
    plt.plot(df['Epoch'], df['Train Loss'], marker='o', label='Train Loss', color='red')
    plt.plot(df['Epoch'], df['Val Loss'], marker='x', label='Val Loss', color='orange')
    plt.title('Training Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    
    # Accuracy / Metrics Plot
    plt.subplot(1, 2, 2)
    plt.plot(df['Epoch'], df['Val Dice'], marker='s', label='Val Dice', color='blue')
    plt.plot(df['Epoch'], df['Val IoU'], marker='^', label='Val IoU', color='green')
    plt.title('Validation Metrics')
    plt.xlabel('Epoch')
    plt.ylabel('Score')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.show()
else:
    print("Henüz log dosyası oluşmamış. Lütfen önce 6. adımdaki eğitimi başlatın.")

## 8) Modeli Test Et (İnference & Görselleştirme)

Təlim etdiyimiz ən yaxşı modeli (`unet_best.pth`) yükləyirik və Test datasetindəki görünməmiş şəkillər üzərində sınaqdan keçiririk. Orijinal peyk şəkli, əsl maska və modelimizin tapdığı binaları yan-yana görəcəksiniz.

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
import random
import glob
import os
from models.unet import UNet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 1. Ən yaxşı modeli yükləyirik
model = UNet(in_channels=3, out_channels=1).to(device)
model.load_state_dict(torch.load('checkpoints/unet_best.pth', map_location=device))
model.eval()

# 2. Test üçün şəkilləri tapırıq
img_paths = sorted(glob.glob('data/mass_tiles_npz/images/*.npz'))
lbl_paths = sorted(glob.glob('data/mass_tiles_npz/labels/*.npz'))

if len(img_paths) == 0:
    print('Test üçün şəkil tapılmadı!')
else:
    # Datasetimizi (building_dataset.py daxilində olduğu kimi) eyni məntiqlə test üçün ayırırıq:
    # 80% Train, 10% Val, 10% Test 
    from sklearn.model_selection import train_test_split
    _, test_imgs, _, test_lbls = train_test_split(img_paths, lbl_paths, test_size=0.2, random_state=42)
    _, test_imgs, _, test_lbls = train_test_split(test_imgs, test_lbls, test_size=0.5, random_state=42)
    
    print(f'Test datasetində {len(test_imgs)} şəkil var.')
    
    # Təsadüfi 3 şəkil seçib yoxlayırıq
    num_samples = min(3, len(test_imgs))
    sample_indices = random.sample(range(len(test_imgs)), num_samples)
    
    plt.figure(figsize=(15, 5 * num_samples))
    
    for i, idx in enumerate(sample_indices):
        # Datşyüklə
        img_arr = np.load(test_imgs[idx])['arr_0']
        mask_arr = np.load(test_lbls[idx])['arr_0']
        
        # Model üçün hazırla (Normalizasiya)
        img_input = img_arr.astype(np.float32) / 255.0 if img_arr.max() > 1 else img_arr.astype(np.float32)
        
        img_tensor = torch.tensor(img_input).unsqueeze(0).to(device)  # [1, 3, 256, 256]
        
        # Proqnoz (Prediction)
        with torch.no_grad():
            preds = model(img_tensor)
            preds = torch.sigmoid(preds)  # Logits -> Probability
            preds = (preds > 0.5).float().cpu().numpy()[0, 0] # [256, 256]
            
        # Original şəkli vizuallaşdırmaq üçün [C, H, W] -> [H, W, C] çevir
        img_display = img_arr.transpose(1, 2, 0)
        
        # Qrafiklər
        plt.subplot(num_samples, 3, i * 3 + 1)
        plt.imshow(img_display)
        plt.title('Orijinal Peyk Şəkli')
        plt.axis('off')
        
        plt.subplot(num_samples, 3, i * 3 + 2)
        plt.imshow(mask_arr, cmap='gray')
        plt.title('Əsl Binalar (Ground Truth)')
        plt.axis('off')
        
        plt.subplot(num_samples, 3, i * 3 + 3)
        plt.imshow(preds, cmap='gray')
        plt.title('Modelin Tapdığı Binalar (Prediction)')
        plt.axis('off')
        
    plt.tight_layout()
    plt.show()
